In [1]:
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
starsw_intersection_flow_5k_path = kagglehub.dataset_download('starsw/intersection-flow-5k')
mariaspasyuk_intersection_correct_path = kagglehub.dataset_download('mariaspasyuk/intersection-correct')

print('Data source import complete.')


Using Colab cache for faster access to the 'intersection-flow-5k' dataset.


100%|██████████| 343/343 [00:00<00:00, 1.19MB/s]

Extracting files...
Data source import complete.


# Часть 3. Ultralitics (3 балла)
**3.1 Выполните transfer learning для решения поставленной задачи используя две модели (YOLO11n и YOLO11l) для детекции:**

**3.1.1 Реализуйте код для загрузки данных, обучения и валидации используя Ultralitics**

**3.1.2 Реализуйте логирование тренировочного лосса и валидационных метрик в tensorboard.**

**3.1.3 Обучите две модели. Выполните валидацию используя Ultralitics для подсчета mAP. Сравните скорость обучения и качество решения задачи. Прокомментируйте полученные результаты.**


In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.7 MB/s eta 0:00:00


In [5]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch import optim

import numpy as np
import pandas as pd
import time
import os

from tqdm import tqdm, tqdm_notebook
from PIL import Image
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device=}")

device=device(type='cuda')


In [5]:
!pip install "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 77.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
open

In [6]:
import yaml
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json
import matplotlib.pyplot as plt
torch.backends.cudnn.benchmark = True
torch.cuda.empty_cache()

def setup_dataset():

    dataset_config = {
        'path': '/kaggle/input/intersection-flow-5k/Intersection-Flow-5K',
        'train': '/kaggle/input/intersection-flow-5k/Intersection-Flow-5K/images/train',
        'val': '/kaggle/input/intersection-flow-5k/Intersection-Flow-5K/images/val',
        'test': '/kaggle/input/intersection-flow-5k/Intersection-Flow-5K/images/test',
        'nc': 8,
        'names': ['vehicle', 'bus', 'bicycle', 'pedestrian',
                 'engine', 'truck', 'tricycle', 'obstacle']
    }

    with open('intersection_flow_updated.yaml', 'w') as f:
        yaml.dump(dataset_config, f, default_flow_style=False)

    return 'intersection_flow_updated.yaml'

In [7]:
from torch.utils.tensorboard import SummaryWriter

def train_yolo(model_name, dataset_yaml, epochs, optimizer, lr, unfreeze_layers, **kwargs):

    model = YOLO(f'{model_name}.pt')

    train_args = {
        'data': dataset_yaml,
        'epochs': epochs,
        'imgsz': kwargs.get('imgsz', 512),
        'batch': kwargs.get('batch', 16),
        'lr0': lr,
        'device': '0' if torch.cuda.is_available() else 'cpu',
        'workers': 2,
        'project': f'{model_name}',
        'name': f'{model_name} {epochs}',
        'exist_ok': True,
        'pretrained': True,
        'freeze': -unfreeze_layers,
        'verbose': True,
        'plots': False,
        'save_period': kwargs.get('save_period', 3),
        'cache': False,
        'optimizer': optimizer,
        'warmup_momentum': kwargs.get('warmup_momentum', 0.8),
        'warmup_bias_lr': kwargs.get('warmup_bias_lr', 0.1),
        'weight_decay': kwargs.get('weight_decay', 5e-4),
        'momentum': kwargs.get('momentum', 0.937),
        'patience': 0,
      }



    start_time = time.time()
    results = model.train(**train_args)

    training_time = time.time() - start_time

    print(f" Обучение {model_name} заняло {training_time:.2f} секунд или {training_time/60:.2f} минут")
    return model, results, training_time

Обучим две модели  YOLO, сравним результаты обучения и метрики на валидации.

Я почитала интернет и так поняла, что в ultralytics yolo есть встроенный тензорборд. Видимо это то, что выводится во время обучения модели и после. Чтобы использовать тензорборд, который из summary writer, мне пришлось бы перезапускать все заново, так как я поздно прочитала, что нужен архив с файлами. Ресурсов гпу и времени у меня уже нет на это.

In [ ]:
dataset_yaml = setup_dataset()

model_n, results_n, time_n = train_yolo('yolo11n', dataset_yaml, epochs=10, optimizer="AdamW", lr=0.01,
                                        unfreeze_layers=2, batch=16, imgsz=512)
torch.cuda.empty_cache()
metrics_n = model_n.val()

print(f"YOLO11n:")
print(f"mAP50-95: {metrics_n.box.map:.4f}")
print(f"mAP50: {metrics_n.box.map50:.4f}")
print(f"Время обучения: {time_n:.2f} сек")

Ultralytics 8.3.222 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=intersection_flow_updated.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=-2, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11n 10, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=0, perspective=0.0, plots=False, pose=

In [ ]:
model_l, results_l, time_l = train_yolo('yolo11l', dataset_yaml, epochs=10,  optimizer="AdamW", lr=0.001,
                                        unfreeze_layers=2, batch=16, imgsz=512)
metrics_l = model_l.val()

print(f"YOLO11l:")
print(f"mAP50-95: {metrics_l.box.map:.4f}")
print(f"mAP50: {metrics_l.box.map50:.4f}")
print(f"Время обучения: {(time_l)/60:.2f} минут")

map_difference = metrics_l.box.map - metrics_n.box.map

print(f"Разница в mAP: {map_difference:.4f}")
print(f"Время обучения YOLO11n: {time_n/60} минут, время обучения YOLO11l: {time_l/60} минут")


Ultralytics 8.3.222 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=intersection_flow_updated.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=-2, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11l 10, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=0, perspective=0.0, plots=False, pose

у yolo11l качество лучше, чему у yolo11n, что было ожидаемо. Разница в mAP50-95 довольно значительная, примерно 0.15. Также более сложная модель yolo11l потребляет примерно в 2 раза больше памяти gpu.

3.2 Для YOLO11n поэкспериментируйте со значениями гиперпараметров (imgsz, epochs, freeze, lr0, momentum, weight_decay и другими опционально). Предлагаем также обратить внимание на параметр save_period.


Эксперимент 1: увеличим imgsz с 512 до 640

In [ ]:
dataset_yaml = setup_dataset()
params_changed = {'name': 'exp1', 'imgsz': 640}
torch.cuda.empty_cache()

model, train_results, train_time = train_yolo('yolo11n', dataset_yaml, epochs=10,  optimizer="AdamW", lr=0.001,
                                        unfreeze_layers=2, batch=16, **params_changed)
metrics = model.val()

result = {'map': metrics.box.map, 'map50': metrics.box.map50, 'training_time': train_time, 'params': params_changed}

print(f"mAP50-95 = {metrics.box.map:.4f}")

Ultralytics 8.3.222 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=intersection_flow_updated.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=-2, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11n 10, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=0, perspective=0.0, plots=False, pose

Эксперимент 2: Попробуем еще увеличить размер изображения.

In [ ]:
dataset_yaml = setup_dataset()
params_changed = {'name': 'exp2', 'imgsz': 720}
torch.cuda.empty_cache()

model, train_results, train_time = train_yolo('yolo11n', dataset_yaml, epochs=10,  optimizer="AdamW", lr=0.001,
                                        unfreeze_layers=2, batch=16, **params_changed)
metrics = model.val()

result = {'map': metrics.box.map, 'map50': metrics.box.map50, 'training_time': train_time, 'params': params_changed}

print(f"mAP50-95 = {metrics.box.map:.4f}")

Ultralytics 8.3.222 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=intersection_flow_updated.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=-2, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=720, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11n 10, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=0, perspective=0.0, plots=False, pose

mAP50-95 = 0.34, метрика стала больше с увеличением размера изображения. Больше увеличивать размер не будем, так как может вылететь по памяти.

Эксперимент 3: увеличим learning rate.

In [ ]:
dataset_yaml = setup_dataset()
params_changed = {'name': 'exp3', 'lr': 0.005}
torch.cuda.empty_cache()

model, train_results, train_time = train_yolo('yolo11n', dataset_yaml, epochs=10,  optimizer="AdamW", imgsz=740,
                                        unfreeze_layers=2, batch=16, **params_changed)
metrics = model.val()

result = {'map': metrics.box.map, 'map50': metrics.box.map50, 'training_time': train_time, 'params': params_changed}
print(f"mAP50-95 = {metrics.box.map:.4f}")

Ultralytics 8.3.222 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=intersection_flow_updated.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=-2, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=740, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11n 10, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=0, perspective=0.0, plots=False, pose

Эксперимент 4: еще увеличим learning rate.

In [ ]:
dataset_yaml = setup_dataset()
params_changed = {'name': 'exp4', 'lr': 0.01}
torch.cuda.empty_cache()

model, train_results, train_time = train_yolo('yolo11n', dataset_yaml, epochs=10,  optimizer="AdamW", imgsz=740,
                                        unfreeze_layers=2, batch=16, **params_changed)
metrics = model.val()

result = {'map': metrics.box.map, 'map50': metrics.box.map50, 'training_time': train_time, 'params': params_changed}
print(f"mAP50-95 = {metrics.box.map:.4f}")

Ultralytics 8.3.222 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=intersection_flow_updated.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=-2, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=740, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11n 10, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=0, perspective=0.0, plots=False, pose=

По сравнению с экспериментом 3, лучше не стало. Возможно из-за бОльшего lr модель перепрыгнула оптимум.

Часть 4. Competition (2 балла)
4.1 Выбейте бейзлайн значение 0.47 mAP50-95 на вале, используя код Ultralitics для вычисления метрик.
4.2 Постарайтесь получить как можно более высокое качество детекции на вале.


In [9]:
dataset_yaml = setup_dataset()
params_changed = {'name': 'yolo11lexp1', 'lr': 0.01}
torch.cuda.empty_cache()

model, train_results, train_time = train_yolo('yolo11l', dataset_yaml, epochs=10,  optimizer="AdamW", imgsz=640,
                                        unfreeze_layers=2, batch=16, **params_changed)
metrics = model.val()

result = {'map': metrics.box.map, 'map50': metrics.box.map50, 'training_time': train_time, 'params': params_changed}
print(f"mAP50-95 = {metrics.box.map:.4f}")

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=intersection_flow_updated.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=-2, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11l 10, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=0, perspective=0.0, plots=False, pose=

В общем, у меня не хватило времени на эксперименты с yolo11l. Лучшее значение метрики получилось у yolo11l в эксперименте выше, где модель сравнивалась с yolo11n. Там значение mAP50-95 получилось примерно 0.41.

In [12]:
def yolo_to_coco_pred(model, val_dataset_path):

    results = model.predict(val_dataset_path, save=False, verbose=False, imgsz=416)

    coco_predictions = []

    for image_id, result in enumerate(results):
        if result.boxes is not None and len(result.boxes) > 0:
            boxes = result.boxes.xywh.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy()

            for i, (box, conf, cls) in enumerate(zip(boxes, confs, classes)):
                x, y, w, h = box
                prediction = {
                    'image_id': image_id,
                    'category_id': int(cls) + 1,
                    'bbox': [float(x - w/2), float(y - h/2), float(w), float(h)],
                    'score': float(conf),
                    'segmentation': []
                }
                coco_predictions.append(prediction)

    predictions_file = 'coco_predictions.json'
    with open(predictions_file, 'w') as f:
        json.dump(coco_predictions, f)

    return predictions_file

In [13]:
yolo_to_coco_pred(model=model, val_dataset_path='/kaggle/input/intersection-flow-5k/Intersection-Flow-5K/images/val')

'coco_predictions.json'